#Transformed drivers data
- Read bronze drivers table
- Keep only the columns required for analytics (Drop url column)
- Standardise column names using snake_case (driverId → driver_id, dateOfbirth → date_of_birth)
- concatenate name.givenname and name.familyname to create a new column driver_name and transform it to title case
- Remove duplicate records
- Transform values of columns locality to Title Case
- Write the transformed data to silver drivers table

In [0]:
%run ../00-common/01.environment.config


In [0]:
%python
bronze_table = f"{catalog_name}.{bronze_schema_name}.drivers"
silver_table = f"{catalog_name}.{silver_schema_name}.drivers"

#Read bronze drivers table

In [0]:
%python
drivers_df = spark.read.table(bronze_table)

In [0]:
%python
display(drivers_df)
       


driverId,name,dateOfBirth,nationality,url,ingestion_timestamp,source_file
abate,"List(carlo, abate)",1932-07-10,italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
abecassis,"List(george, abecassis)",1913-03-21,british,http://en.wikipedia.org/wiki/George_Abecassis,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
acheson,"List(kenny, acheson)",1957-11-27,british,http://en.wikipedia.org/wiki/Kenny_Acheson,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
adams,"List(philippe, adams)",1969-11-19,belgian,http://en.wikipedia.org/wiki/Philippe_Adams,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
ader,"List(walt, ader)",1913-12-15,american,http://en.wikipedia.org/wiki/Walt_Ader,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
adolff,"List(kurt, adolff)",1921-11-05,german,http://en.wikipedia.org/wiki/Kurt_Adolff,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
agabashian,"List(fred, agabashian)",1913-08-21,american,http://en.wikipedia.org/wiki/Fred_Agabashian,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
ahrens,"List(kurt, ahrens)",1940-04-19,german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr.",2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
aitken,"List(jack, aitken)",1995-09-23,british,http://en.wikipedia.org/wiki/Jack_Aitken,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
albers,"List(christijan, albers)",1979-04-16,dutch,http://en.wikipedia.org/wiki/Christijan_Albers,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json


#Keep only the columns required for analytics (Drop url column)

In [0]:
%python
from pyspark.sql import functions as F
drivers_dropped_df = drivers_df.drop(F.col("url"))

#Standardise column names using snake_case (driverId → driver_id, dateOfbirth → date_of_birth)
#Rename columns to make them more meaningful (dateOfBirth->date_of_birth)

In [0]:
%python
drivers_renamed_df = (drivers_dropped_df.withColumnRenamed("driverId", "driver_id")
.withColumnRenamed("dateOfBirth", "date_of_birth")
)

#concatenate name.givenname and name.familyname to create a new column driver_name and transform it to title case

In [0]:
%python
drivers_concatenated_df = (drivers_renamed_df
                           .withColumn("driver_name", 
                            F.initcap(F.concat_ws(" ", F.col("name.givenname"), F.col("name.familyname"))))
                           .drop("name")
                           )


#Remove duplicate records

In [0]:
%python
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])

#Transform values of columns driver_name to Title Case

In [0]:
%python
from pyspark.sql import functions as F
drivers_final_df = (drivers_distinct_df
                     .withColumn("nationality", F.initcap(F.col("nationality")))
                   

)

In [0]:
%python
display(drivers_final_df)

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,driver_name
Cannoc,1933-06-21,Canadian,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,John Cannon
Changy,1922-02-05,Belgian,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Alain De Changy
abate,1932-07-10,Italian,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Carlo Abate
abecassis,1913-03-21,British,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,George Abecassis
acheson,1957-11-27,British,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kenny Acheson
adamich,1941-10-03,Italian,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Andrea De Adamich
adams,1969-11-19,Belgian,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Philippe Adams
ader,1913-12-15,American,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Walt Ader
adolff,1921-11-05,German,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Adolff
agabashian,1913-08-21,American,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Fred Agabashian


#Write the transformed data to silver drivers table

In [0]:
%python
(
    drivers_final_df
    .write
    .format("delta") 
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
%python
display(spark.table(silver_table))

season,round,race_name,race_date,circuit_id,ingestion_timestamp,source_file
1950,1,British Grand Prix,1950-05-13,silverstone,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,2,Monaco Grand Prix,1950-05-21,monaco,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,3,Indianapolis 500,1950-05-30,indianapolis,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,4,Swiss Grand Prix,1950-06-04,bremgarten,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,5,Belgian Grand Prix,1950-06-18,spa,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,6,French Grand Prix,1950-07-02,reims,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,7,Italian Grand Prix,1950-09-03,monza,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,1,Swiss Grand Prix,1951-05-27,bremgarten,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,2,Indianapolis 500,1951-05-30,indianapolis,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,3,Belgian Grand Prix,1951-06-17,spa,2026-06-15T21:47:54.299134Z,dbfs:/Volumes/formula1/landing/files/races.csv
